<a href="https://colab.research.google.com/github/luigidaniel1113-cpu/Sistemas-Operativo/blob/main/Echeverria%20Delgado%20Luis%20Daniel_Practica1_Conceptos_SO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💻 Sistemas Operativos - Exploración de la Estructura del SO y Recursos del Sistema
**Asignatura:** Fundamentos y Diseño de Sistemas Operativos (6.º Semestre)    
**Unidad I:** Conceptos y Evolución de los Sistemas Operativos  
**Tiempo estimado:** 2 horas (Asíncrono)  
**Repositorio Base:** Integración con GitHub  

---

## 1. Objetivos de Aprendizaje
1. Identificar la estructura física y lógica de la máquina virtual Linux subyacente en Google Colab.
2. Analizar el rol del Kernel en la administración de recursos hardware (CPU, Memoria, E/S).
3. Ejecutar y comprender llamadas al sistema (Syscalls) y la arquitectura en modo Kernel/Usuario.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede a tu entorno de **NotebookLM del Curso**, abre la sección de la *Unidad I* y consulta:
---

- *¿Qué es el Kernel de Linux y cómo abstrae el hardware?*  
**RESPUESTA:** El Kernel de Linux es la capa de software más cercana al hardware y constituye la parte central del sistema operativo

Se ejecuta en modo kernel (también conocido como modo supervisor o protegido), lo que le otorga acceso completo a todos los recursos físicos y la capacidad de ejecutar cualquier instrucción de la máquina.

Sus características y funciones principales incluyen:
- Residencia en memoria: Se mantiene permanentemente en la memoria principal como software residente.

- En los sistemas Linux, este núcleo reside físicamente en el archivo vmlinuz.

- Administración de recursos: Controla el acceso de la CPU a los dispositivos de entrada/salida (E/S) y a la unidad de administración de memoria, encargándose de la planificación de la CPU, la gestión de memoria, la creación y terminación de procesos y el manejo de señales.

- Estructura monolítica pero modular: Aunque posee un diseño monolítico clásico
, Linux logra una gran flexibilidad mediante una arquitectura de módulos cargables. Esto le permite incorporar o retirar funcionalidades de forma dinámica en tiempo de ejecución (como controladores de nuevos dispositivos o sistemas de archivos) sin necesidad de recompilar el núcleo o reiniciar la computadora

**¿Cómo abstrae el hardware?**

Para ocultar la complejidad y las "interfaces feas" de los componentes físicos, presentando en su lugar abstracciones simples, elegantes y uniformes a las aplicaciones, el kernel de Linux implementa los siguientes mecanismos:

- **Abstracción de dispositivos como archivos especiales**: Siguiendo la filosofía de diseño de Unix, Linux integra los dispositivos de hardware directamente en el sistema de archivos, representándolos como archivos especiales (usualmente ubicados en el directorio /dev). De esta forma, las aplicaciones no requieren conocer los detalles de bajo nivel del hardware y pueden utilizar las mismas llamadas al sistema estándares (open, read, write y close) para interactuar de forma uniforme tanto con un archivo en disco como con un teclado, terminal o impresora.

- **Controladores de dispositivos (Drivers)**: El kernel utiliza controladores de dispositivos para aislar al resto del sistema operativo de las idiosincrasias y particularidades de cada hardware físico. Al proveer interfaces estándar entre los drivers y el resto del sistema, la mayor parte del software de E/S se ejecuta de manera independiente de la máquina. Estos controladores se clasifican abstractamente en dispositivos de caracteres (flujos lineales de bytes sin búsquedas, como terminales) y dispositivos de bloques (que permiten acceso aleatorio y búsquedas de bloques de tamaño fijo, como discos).

- **Sistema de Archivos Virtual (VFS) y Capa de Bloque Genérica**: El VFS define abstracciones comunes del sistema de archivos, permitiendo que múltiples formatos de almacenamiento (locales y de red) coexistan de manera transparente para el usuario. Directamente por debajo, una capa de dispositivos de bloques genéricos ofrece una interfaz común que oculta las espantosas diferencias arquitectónicas y geométricas (sectores, cilindros, pistas) de los dispositivos de almacenamiento físico respecto a las implementaciones de los sistemas de archivos.

- **Memoria Virtual y MMU**: En cooperación con la Unidad de Administración de Memoria (MMU) del procesador, el kernel gestiona y abstrae la memoria física del sistema implementando un esquema de memoria virtual. Traduce en tiempo de ejecución las direcciones lógicas utilizadas por los programas a las direcciones físicas reales de la RAM, proporcionando un mapa de memoria independiente para cada proceso y evitando que interfieran entre sí o comprometan el espacio del propio kernel.

- **Planificación de CPU y Tareas**: Abstrae la CPU física multiplexando su uso para dar la ilusión de que existen múltiples "procesadores virtuales" ejecutándose de forma concurrente. En Linux no se hace distinción interna entre procesos e hilos; el kernel los trata de manera uniforme como tareas (task_struct). Cuando ocurre una interrupción o se agota el quántum de tiempo, el kernel guarda el estado de los registros del procesador (contexto de registros) de la tarea en ejecución y restaura el de la siguiente, de modo que el cambio de flujo de ejecución sea invisible y seguro.

- *¿Qué función cumple la jerarquía de memoria en un SO moderno?*  
**RESPUESTA:** La jerarquía de memoria en un sistema informático moderno tiene como función principal resolver un problema técnico y económico fundamental: los desarrolladores y usuarios desearían contar con una memoria privada que fuera infinitamente rápida, infinitamente grande, no volátil y sumamente barata.

Como ninguna tecnología física actual satisface todos estos objetivos simultáneamente, la solución consiste en organizar los componentes de almacenamiento en múltiples niveles o capas de diferente velocidad, capacidad y costo.

Al descender por los niveles de la jerarquía (desde los registros del procesador y la memoria caché, pasando por la memoria principal o RAM, hasta el almacenamiento secundario como discos y cintas) ocurren de manera predecible los siguientes efectos:

- Disminución del costo por bit.

- Aumento de la capacidad de almacenamiento.

- Aumento del tiempo de acceso (la memoria se vuelve más lenta).

- Disminución de la frecuencia de acceso a la memoria por parte del procesador.

**Funciones clave en un Sistema Operativo moderno**

Para explotar esta estructura física de manera óptima, el sistema operativo (SO), en estrecha colaboración con el hardware, realiza las siguientes funciones fundamentales:

1. Mitigar la brecha de rendimiento de la CPU (El cuello de botella de von Neumann)

La velocidad de cálculo de los procesadores ha crecido exponencialmente más rápido en comparación con la velocidad de acceso a las memorias principales. Para evitar que la CPU se mantenga inactiva esperando instrucciones o datos del almacenamiento físico lento, la jerarquía de memoria interpone pequeños bloques de memoria ultrarrápida (como registros y cachés L1, L2 y L3) que alimentan al procesador a su velocidad nativa.

2. Explotar el principio de proximidad de referencias (Localidad)
La viabilidad técnica de toda la jerarquía se sostiene sobre el principio de proximidad de referencias, el cual postula que durante la ejecución de un programa, los accesos a memoria (instrucciones y datos) tienden a agruparse en zonas muy delimitadas en un intervalo de tiempo corto.

Proximidad temporal: Si se hace referencia a una posición de memoria, es muy probable que se acceda a ella de nuevo en un futuro cercano (como variables dentro de un bucle).

Proximidad espacial: Si se accede a una dirección, es muy probable que se acceda a direcciones físicamente cercanas en breve (como la ejecución secuencial de instrucciones o el recorrido de arrays).

El SO y el hardware de control aprovechan este comportamiento trayendo bloques de información desde los niveles lentos hacia los niveles más rápidos de manera anticipada o bajo demanda. Esto permite que la mayor parte de las peticiones se resuelvan en los niveles más veloces (un acierto en caché o en memoria principal), logrando un tiempo de acceso efectivo sumamente rápido.
3. Soportar y abstraer la Memoria Virtual
Es tarea del sistema operativo abstraer esta jerarquía física y presentar a las aplicaciones un modelo uniforme de memoria útil, protegiendo al mismo tiempo a los procesos entre sí. Mediante la memoria virtual, el SO desacopla las direcciones lógicas que genera el procesador de las direcciones físicas reales de la RAM. Esto permite:
Ejecutar programas más grandes que la memoria RAM física disponible.

Mantener en la memoria principal (como un conjunto residente) únicamente las páginas o segmentos del proceso que se necesitan para la ejecución inmediata, mientras que el resto se conserva en el dispositivo de almacenamiento secundario (intercambio o swap).
Optimizar la asignación dinámica de memoria y aumentar de forma eficiente el grado de multiprogramación del sistema.

4. Gestión de la migración y la coherencia de datos
El sistema operativo se encarga de controlar de forma transparente el flujo de la información que viaja entre los distintos niveles. Cuando los datos se modifican en una capa superior y rápida (por ejemplo, en un búfer o una página de la memoria principal), se genera una falta de consistencia con las copias en las capas inferiores. El SO aplica políticas de actualización (como escrituras inmediatas o diferidas) para transferir y limpiar estas porciones hacia el almacenamiento permanente antes de que se desalojen de la memoria principal o ante fallos del sistema.

---

## 3. Sección Práctica 1: Inspección de Recursos del Sistema (Bash / CLI)

En esta sección exploraremos el sistema sobre el cual se ejecuta este Cuaderno (Servidor Linux provisto por Google).

### [CÓDIGO 1.1] Identificación del Kernel y la Distribución Linux
```bash
# Ejecutamos comandos Bash usando el prefijo !
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

In [ ]:
!echo "=== INFORMACIÓN DEL KERNEL ==="
!uname -a
!echo ""
!echo "=== DISTRIBUCIÓN DE LINUX ==="
!cat /etc/os-release | grep -E "PRETTY_NAME|VERSION"

=== INFORMACIÓN DEL KERNEL ===
Linux db7fa64c419b 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux

=== DISTRIBUCIÓN DE LINUX ===
PRETTY_NAME="Ubuntu 22.04.5 LTS"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy


### [CÓDIGO 1.2] Inspección de la CPU (Recurso Procesador)
```bash
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

In [ ]:
!echo "=== ARQUITECTURA Y NÚCLEOS DE CPU ==="
!lscpu | grep -E "Model name|CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Architecture"

=== ARQUITECTURA Y NÚCLEOS DE CPU ===
Architecture:                            x86_64
CPU(s):                                  2
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Core(s) per socket:                      1
NUMA node0 CPU(s):                       0,1


### [CÓDIGO 1.3] Inspección de la Memoria RAM y Jerarquía de Almacenamiento
```bash
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

In [ ]:
!echo "=== ESTADO DE LA MEMORIA RAM (en MB) ==="
!free -m
!echo ""
!echo "=== SISTEMA DE ARCHIVOS Y DISCO DURA ==="
!df -h /

=== ESTADO DE LA MEMORIA RAM (en MB) ===
               total        used        free      shared  buff/cache   available
Mem:           12975         759        8980           2        3235       11933
Swap:              0           0           0

=== SISTEMA DE ARCHIVOS Y DISCO DURA ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


### Pregunta de Análisis 1 (Responde en la celda de texto):
Basándote en las lecturas de Tanenbaum en NotebookLM y en los resultados obtenidos arriba:

1. ¿Qué versión de Kernel Linux está ejecutando la máquina virtual de Colab?  
**RESPUESTA:** 6.6.122+

2. ¿Cuántos núcleos de procesador y cuánta memoria RAM tiene asignados tu entorno virtual?  
**RESPUESTA:** Nucelos de procesador 2
Memoria RAM 12975 MB

## 4. Sección Práctica 2: Abstracción de Hardware a través del Sistema `/proc`

En Linux, `/proc` es un **pseudo-sistema de archivos**. No ocupa espacio en disco; es una interfaz directa mediante la cual el Kernel expone información sobre el estado de los recursos y procesos del sistema.

### [CÓDIGO 2.1] Exploración del sistema /proc
```bash
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

In [ ]:
!echo "=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ==="
!head -n 10 /proc/meminfo
!echo ""
!echo "=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ==="
!cat /proc/uptime

=== INFORMACIÓN DETALLADA DE LA MEMORIA DESDE KERNEL ===
MemTotal:       13286944 kB
MemFree:         9192764 kB
MemAvailable:   12223144 kB
Buffers:          136496 kB
Cached:          3091192 kB
SwapCached:            0 kB
Active:           484156 kB
Inactive:        3408952 kB
Active(anon):       1632 kB
Inactive(anon):   666172 kB

=== UPTIME DEL SISTEMA (Segundos activo / Segundos inactivo) ===
1423.20 2587.71


### [CÓDIGO 2.2] Visualización del árbol de llamadas e interrupciones del hardware
```bash
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

In [ ]:
!echo "=== INTERRUPCIONES DE HARDWARE REGISTRADAS ==="
!head -n 20 /proc/interrupts

=== INTERRUPCIONES DE HARDWARE REGISTRADAS ===
           CPU0       CPU1       
  0:        259          0   IO-APIC   0-edge      timer
  1:          0          9   IO-APIC   1-edge      i8042
  4:          0       2714   IO-APIC   4-edge      ttyS0
  8:          0          0   IO-APIC   8-edge      rtc0
  9:          0          0   IO-APIC   9-fasteoi   acpi
 10:       1544          0   IO-APIC  10-fasteoi   virtio2
 12:          3          0   IO-APIC  12-edge      i8042
 24:          0          0  PCI-MSIX-0000:00:06.0   0-edge      virtio3-config
 25:          0         37  PCI-MSIX-0000:00:06.0   1-edge      virtio3-input
 26:          0          0  PCI-MSIX-0000:00:03.0   0-edge      virtio0-config
 27:          0          0  PCI-MSIX-0000:00:03.0   1-edge      virtio0-control
 28:          0          0  PCI-MSIX-0000:00:03.0   2-edge      virtio0-event
 29:          0     155269  PCI-MSIX-0000:00:03.0   3-edge      virtio0-request
 30:          0          0  PCI-MSIX-0000:00:0

## 5. Sección Práctica 3: Primer Contacto con Bajo Nivel (C / Syscalls POSIX)

A continuación, crearemos, compilaremos y ejecutaremos un programa en C para observar directamente las **Llamadas al Sistema** (`sys_write`, `sys_getpid`) que interactúan entre el **Modo Usuario** y el **Modo Kernel**.

```bash
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

In [ ]:
%%writefile syscall_demo.c
/*
 * Práctica 1 - Introducción a Llamadas al Sistema POSIX
 * Este programa realiza llamadas al sistema para obtener datos del proceso.
 */

# include <stdio.h>
# include <unistd.h>
# include <sys/types.h>

int main() {
    // getpid() es una llamada al sistema POSIX para solicitar al Kernel el ID del proceso actual
    pid_t process_id = getpid();
    pid_t parent_id = getppid();

    printf("--------------------------------------------------\n");
    printf("[MODO USUARIO] Ejecutando programa en C...\n");
    printf("[LLAMADA AL SISTEMA] PID del Proceso Actual: %d\n", process_id);
    printf("[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): %d\n", parent_id);
    printf("--------------------------------------------------\n");

    return 0;
}

Overwriting syscall_demo.c


### [CÓDIGO 3.2] Compilación y Ejecución del Código C usando GCC
```bash
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

In [ ]:
!gcc -Wall syscall_demo.c -o syscall_demo
!./syscall_demo

--------------------------------------------------
[MODO USUARIO] Ejecutando programa en C...
[LLAMADA AL SISTEMA] PID del Proceso Actual: 7537
[LLAMADA AL SISTEMA] PID del Proceso Padre (Bash/Python): 2935
--------------------------------------------------


### [CÓDIGO 3.3] Rastreando las llamadas al sistema reales con `strace`
```bash
# Instalamos strace para interceptar las llamadas que hace el ejecutable al Kernel
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

In [ ]:
!apt-get install -y strace > /dev/null 2>&1
!strace -e trace=write,getpid ./syscall_demo

strace: Cannot stat './syscall_demo': No such file or directory


## 6. Desafío Asíncrono / Entregable de la Unidad I

### Instrucciones:
1. Modifica el archivo `syscall_demo.c` de la Sección 5 creando un nuevo archivo `desafio_u1.c` mediante celdas mágicas `%%writefile`.
2. Agrega la llamada al sistema POSIX `getuid()` (para obtener el ID del usuario que ejecuta el proceso) y la función `sleep(5)` para pausar la ejecución durante 5 segundos.
3. Ejecuta el programa en segundo plano mediante Bash y utiliza el comando `ps aux` para capturar el proceso en la lista de procesos activos mientras duerme.
4. Vuelve a ejecutar `strace` sobre el nuevo binario para identificar las syscalls `nanosleep` y `getuid`.

---

In [7]:
%%writefile desafio_u1.c
#include <stdio.h>
#include <unistd.h>
#include <sys/types.h>

int main() {
    // Obtener e imprimir el ID del usuario
    uid_t uid = getuid();
    printf("ID del usuario (UID): %d\n", uid);
    fflush(stdout);

    // Pausar la ejecución durante 5 segundos
    printf("Pausando ejecución por 5 segundos...\n");
    fflush(stdout);
    sleep(5);

    printf("Fin del programa.\n");
    return 0;
}

Overwriting desafio_u1.c


In [8]:
%%bash
gcc desafio_u1.c -o desafio_u1
./desafio_u1 &
sleep 1
ps aux | grep desafio_u1

ID del usuario (UID): 0
Pausando ejecución por 5 segundos...
root       23066  0.0  0.0   2684  1780 ?        S    01:42   0:00 ./desafio_u1
root       23073  0.0  0.0   6548  2360 ?        S    01:42   0:00 grep desafio_u1
Fin del programa.


In [9]:
%%bash
apt-get install -y strace > /dev/null 2>&1
cd /content
strace -e getuid,nanosleep ./desafio_u1

ID del usuario (UID): 0
Pausando ejecución por 5 segundos...
Fin del programa.


getuid()                                = 0
+++ exited with 0 +++


### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas incluyendo las respuesta a las preguntas (`Archivo` -> `Guardar una copia en Drive`).
2. Sube los cambios a tu repositorio de GitHub de la materia bajo el nombre `JUAN PEREZ Practica1_Conceptos_SO.ipynb` usando la integración directa (`Archivo` -> `Guardar una copia en GitHub`).
3. Adjunta en tu entregable de plataforma Moodle, el enlace de GitHub.